In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
from langchain_tavily import TavilySearch

web_search = TavilySearch(
    max_results=5,
    topic="general",
)

In [3]:
from langchain_openai import ChatOpenAI
import os
from langchain.messages import HumanMessage
model = ChatOpenAI(
    model="qwen3.6-flash",
    base_url=os.getenv("BAILIAN_BASE_URL"),
    api_key=os.getenv("BAILIAN_API_KEY")
)

In [4]:
# 定义记忆管理
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
connection = sqlite3.connect(
    "../../resources/checkpoint.db",
    check_same_thread=False
)
checkpointer = SqliteSaver(connection)
checkpointer.setup()

In [5]:
from langchain.agents import create_agent

system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
"""

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

In [6]:
from langchain.messages import HumanMessage

# 准备多模态消息，图片是网络上的冰箱食物图片
image_url = "https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg"

multimodal_message = HumanMessage(
    content=[
        {
            "type": "image_url",
            "image_url": {
                "url": image_url
            }
        },
        {
            "type": "text",
            "text": "帮我看看这些食材能做些什么？"
        }
    ]
)

config = {
    "configurable": {
        "thread_id": "6"
    }
}

response = agent.invoke(
    {
        "messages": [multimodal_message]
    },
    config
)

In [8]:
print(response)

{'messages': [HumanMessage(content=[{'type': 'image_url', 'image_url': {'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}], additional_kwargs={}, response_metadata={}, id='3515aea7-8496-4d0b-8bf4-a65609905aee'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 309, 'prompt_tokens': 2273, 'total_tokens': 2582, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 156, 'rejected_prediction_tokens': None, 'text_tokens': 153}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'image_tokens': 262, 'text_tokens': 2011}}, 'model_provider': 'openai', 'model_name': 'qwen3.6-flash', 'system_fingerprint': None, 'id': 'chatcmpl-f2687e1c-3ddb-9737-a430-df758eed6c57', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f9d28-76ef-70d0-a45d-0446d59

In [9]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'image_url', 'image_url': {'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_9b4d4575594c4ae18a045c37)
 Call ID: call_9b4d4575594c4ae18a045c37
  Args:
    query: 生菜 红彩椒 小番茄 蘑菇 三文鱼 鸡胸肉 西兰花 洋葱 可行菜谱
    search_depth: basic
    topic: general
    include_images: False
    include_domains: []
    exclude_domains: []
    time_range: 
    start_date: 
    end_date:
================================= Tool Message =================================
Name: tavily_search

Error invoking tool 'tavily_search' with kwargs {'query': '生菜 红彩椒 小番茄 蘑菇 三文鱼 鸡胸肉 西兰花 洋葱 可行菜谱', 'search_depth': 'basic', 'topic': 'general', 'include_images': 'False', 'include_domains': '[]', 'exclude_domains': '[]', 'time_ran

In [10]:
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢第1道菜，可以说的更详细点吗？")]},
    config
)

In [16]:
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

# 香煎三文鱼配时蔬沙拉（详细版）


### 一、食材清单（以2人份为例）  
- **主料**：三文鱼排2块（约300g）、生菜8片、小番茄6颗  
- **辅料**：红彩椒半个（切丝）、洋葱1/4个（切圈）、西兰花5朵（需焯水）  
- **调料**：橄榄油、海盐、黑胡椒碎、柠檬汁（少量）  


### 二、制作步骤（附实操细节）  
1. **三文鱼预处理**  
   - 用厨房纸**彻底吸干三文鱼表面水分**（关键！防止煎制时油溅且保证鱼皮酥脆）→ 两面均匀抹上海盐、黑胡椒，静置腌10分钟入味。  

2. **蔬菜处理**  
   - 生菜洗净后**沥干水分**（建议用 salad spinner 甩干，避免沙拉过于湿软）→ 小番茄对半切；红彩椒、洋葱分别切成细长丝；西兰花焯水（沸水+少许盐，煮1分钟后立刻捞出冲凉水，锁住翠绿脆感）。  

3. **煎三文鱼（核心环节）**  
   - 平底锅中火烧热→ 倒入橄榄油（约1勺，量需覆盖锅底）至微微冒烟 → **鱼皮朝下**入锅（先煎鱼皮能让口感更丰富，若不喜欢鱼皮可省略此步骤直接煎鱼肉面）→ 保持中火煎3分钟 → 翻面后再煎2分钟（期间可用筷子轻戳鱼肉，能轻易穿透且无血色渗出即代表熟透）。  

4. **组装与调味**  
   - 将生菜、小番茄、彩椒丝、洋葱圈、西兰花分层铺在盘中 → 摆上煎好的三文鱼 → 淋少许橄榄油提香 + 挤几滴柠檬汁中和油腻感 → 最后撒海盐和黑胡椒碎点睛即可开吃。  


### 三、营养&口感双重加分项  
- **蛋白质来源优化**：三文鱼的 Omega-3 脂肪酸有助于保护心血管，而后续若冰箱中有鸡胸肉，可替换成“香煎鸡胸肉版”强化增肌需求~  
- **蔬菜多样性设计**：生菜的清爽、小番茄的酸甜、彩椒的脆甜、洋葱的微辛、西兰花的绵密……多种口感碰撞让每口都充满新鲜感。  


### 四、为什么它排第一？（多维度评估）  
- **制作难度**：★★☆☆☆（仅需掌握“煎鱼控火候”1个技巧，新手5分钟内就能get）  
- **营养价值**：★★★★★（涵盖优质蛋白+高维C蔬菜+膳食纤维三重营养矩

In [17]:
# 流式调用
from langchain.messages import AIMessageChunk

for chunk,metadata in agent.stream({"messages":[multimodal_message]},config,stream_mode='messages'):
    if isinstance(chunk,AIMessageChunk) and chunk.content:
        print(chunk.content,end="",flush=True)


没问题！这确实是一个非常明智的选择，三文鱼富含Omega-3脂肪酸，肉质鲜嫩，搭配清爽的蔬菜正好平衡了口感。

以下是为您定制的**“香煎三文鱼配时蔬沙拉”**的详细实操指南。即使是厨房新手，也能做出餐厅级的卖相和味道。

---

### 🥗 食谱：香煎三文鱼配缤纷时蔬沙拉

#### 🛒 一、 备料清单（基于您的冰箱库存）

*   **主角**：三文鱼排（图片中第二层左侧）—— 这是整道菜的灵魂。
*   **配角**：
    *   **绿色担当**：西兰花（第一层左侧） + 生菜（第一层及底层抽屉都有）。
    *   **色彩点缀**：红彩椒（顶层右侧）+ 小番茄（顶层右侧玻璃碗）。
    *   **提味辅料**：洋葱（底层右侧的大白球状物体）。
*   **调料柜常备**：橄榄油、海盐、黑胡椒、柠檬（如果有）、蜂蜜或醋（可选）。

---

#### 👨‍ 二、 烹饪步骤详解

**第一步：处理食材**（Mise en place）
1.  **三文鱼预处理**（关键一步）
    *   拿出厨房纸巾，**彻底吸干**三文鱼表面的水分。
    *   *厨师心法*：这一步非常重要！只有表面干燥，下锅时才能发生“美拉德反应”，煎出金黄酥脆的外皮，而不是软绵绵地被煮熟。
2.  **蔬菜处理**：
    *   西兰花洗净，切成一口大小的小朵。
    *   小番茄对半切开；红彩椒切丝；洋葱切细圈或碎末。
    *   生菜沥干水分备用。

**第二步：煎制三文鱼**（核心环节）
1.  **腌制**：在擦干水分的鱼肉两面均匀撒上少许海盐和黑胡椒，静置5分钟入味。
2.  **热锅**：平底锅开**中大火**烧热，倒入一勺橄榄油。待油微微冒烟（这说明温度够了）。
3.  **下锅**：将三文鱼**鱼皮朝下**放入锅中。
    *   *注意*：刚放进去不要动它！保持中火煎约 **3-4分钟**，你会看到鱼肉侧面从底部慢慢变白。这能保证皮脆肉嫩。
4.  **翻面**：翻面后继续煎 **2-3分钟**。
    *   *判断生熟*：用筷子轻轻一按鱼肉最厚处，如果感觉有弹性且不散开，或者内部呈现粉嫩的橙色但已凝固，即可出锅。不要煎过头，否则口感会变柴。

**第三步：制作配菜**（快速快手菜）
1.  **焯水西兰花**：烧一锅开水，水里

In [18]:
# 获取会话历史
for m in checkpointer.get(config)['channel_values']['messages']:
    print(type(m))
    print(m)

<class 'langchain_core.messages.human.HumanMessage'>
content=[{'type': 'image_url', 'image_url': {'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}] additional_kwargs={} response_metadata={} id='3515aea7-8496-4d0b-8bf4-a65609905aee'
<class 'langchain_core.messages.ai.AIMessage'>
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 309, 'prompt_tokens': 2273, 'total_tokens': 2582, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 156, 'rejected_prediction_tokens': None, 'text_tokens': 153}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'image_tokens': 262, 'text_tokens': 2011}}, 'model_provider': 'openai', 'model_name': 'qwen3.6-flash', 'system_fingerprint': None, 'id': 'chatcmpl-f2687e1c-3ddb-9737-a430-df758eed6c57', 'finish_reason': 'tool_calls', 'logpr